# Enzyme Design Pipeline — Course 27666

```
RFD3  →  LigandMPNN  →  RF3
```

Design *de novo* enzymes around a fixed catalytic motif (a *theozyme*), then validate
them in silico. Three stages:

1. **RFD3 (RFdiffusion3)** — generate enzyme *backbones* by diffusion, holding the
   catalytic residues in their theozyme geometry while the surrounding scaffold is built
   from scratch around the substrate.
2. **LigandMPNN** — design amino-acid *sequences* onto each backbone, conditioned on the
   small-molecule substrate and keeping the catalytic residues fixed.
3. **RF3 (RoseTTAFold3)** — *fold & score* each enzyme + substrate complex to estimate how
   confident the fold is and how well-defined the substrate pocket is.

The worked example designs a **serine hydrolase** around a **PET-monomer-mimic substrate**
(theozyme `5XH3`, bundled in `inputs/`). Everything runs on the GPU cluster via the
**`c27666`** LSF queue.

## How to run this notebook

**Read `README_Enzyme_design.md` first** for full setup. In short:

1. Clone this repo into *your* scratch space: `cd /work3/$USER && git clone <repo-url>`
2. Register the shared conda env as a Jupyter kernel once (see README), select it.
3. Run the cells top to bottom. GPU stages **don't run in the notebook** — each writes an
   LSF submit script and prints a `bsub < ...` command. You run that in a terminal, wait
   for the job to finish (`bstat`), then continue with the next cell.

**Queue notes (`c27666`):** shared by the whole class on only a couple of GPUs, so keep
the design counts small (defaults below are deliberately tiny). Max wall time is 12 h and
the queue default is 15 min — the submit scripts always set `-W` explicitly.

## Setup: imports, repo paths, working directories

In [ ]:
# --- Core ---
import os, sys, re, glob, json, math, shutil, csv
from pathlib import Path
from copy import deepcopy

# --- Data / plotting ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Locate the repo root (the folder containing this notebook) ---
# Jupyter starts in the notebook's directory, so cwd is the repo root.
REPO_ROOT = Path.cwd()
if not ((REPO_ROOT / "inputs").is_dir() and (REPO_ROOT / "lib").is_dir()):
    raise RuntimeError(
        f"cwd={REPO_ROOT} is not the repo root (no inputs/ and lib/).\n"
        "Set it manually, e.g.:  REPO_ROOT = Path('/work3/' + os.environ['USER'] + '/27666_Protein_Design/Day_7')"
    )

# Make the bundled helper modules importable
sys.path.insert(0, str(REPO_ROOT / "lib"))
import jupyter_utils
from rf3_metrics import gather_rf3_metrics

print("Repo root:", REPO_ROOT)
print("User:", os.environ.get("USER", "?"))

In [ ]:
# All run outputs go under work/<experiment>/ inside your clone (git-ignored).
experiment = "enz_01"          # bump this for each new design round
WORK = REPO_ROOT / "work" / experiment

_subdirs = ["cmds", "submit", "logs", "configs", "scores",
            "diffusion_out", "mpnn_out", "rf3_out", "best_designs"]
for d in _subdirs:
    (WORK / d).mkdir(parents=True, exist_ok=True)

cmds_dir          = str(WORK / "cmds")
submit_dir        = str(WORK / "submit")
logs_dir          = str(WORK / "logs")
configs_dir       = str(WORK / "configs")
scores_dir        = str(WORK / "scores")
diffusion_out_dir = str(WORK / "diffusion_out")
mpnn_out_dir      = str(WORK / "mpnn_out")
rf3_out_dir       = str(WORK / "rf3_out")
best_designs_dir  = str(WORK / "best_designs")

pd.set_option("display.max_columns", None)
print("Working dir:", WORK)

## 1. RFD3 — generate enzyme backbones

RFD3 diffuses an enzyme backbone from noise while **holding the catalytic residues fixed**
in their theozyme geometry and building the scaffold around the substrate. Four things
define the job:

- **`contig`** — the scaffold layout. Free ranges (e.g. `30-50`) are designed from scratch;
  fixed entries like `A131-132` copy those motif residues straight from the input PDB so
  their geometry is preserved; `/0` separates chains.
- **`catres`** — the catalytic residues (here the Ser–Asp–His triad). They are kept in
  place and, later, **not** redesigned by LigandMPNN.
- **`fixed_atoms`** — the specific sidechain atoms on each catalytic residue to pin during
  diffusion, so the catalytic geometry is held exactly.
- **`ligand`** — the substrate residue name in the input PDB (`pt1` here). RFD3 builds the
  scaffold around it.

Unlike binder design there are no hotspots and `infer_ori_strategy` is left as `None`.

### Build the RFD3 input JSON

In [ ]:
design_name = "rfd3_5xh3"
input_pdb   = str(REPO_ROOT / "inputs" / "5XH3.pdb")   # serine hydrolase theozyme, ligand pt1

# Scaffold layout: free ranges (designed) interspersed with fixed motif residues; /0 = chain break
contig = "30-50,A58-58,40-60,A131-132,20-40,A177-177,20-40,A208,40-60"
length = "180-250"                 # total design length range (residues)
redesign_motif_sidechains = False
is_non_loopy              = True   # bias toward helices, less loopy backbones
infer_ori_strategy        = None   # None for enzyme design ("hotspots" is for binders)

ligand_name = "pt1"                # must match the ligand residue name in input_pdb

# Catalytic residues (Ser-Asp-His triad) — LigandMPNN will not redesign these
catres = ["A131", "A177", "A208"]

# Key sidechain atoms to pin during diffusion (preserves catalytic geometry)
fixed_atoms = {
    "A131": "OG,CA,CB",            # SER nucleophile
    "A177": "OD1,OD2,CG,CB",       # ASP
    "A208": "NE2,ND1,CE1,CD2,CG",  # HIS
}

rfd3_json = str(Path(configs_dir) / "rfd3_input.json")

payload = {
    design_name: {
        "dialect": 2,
        "input": input_pdb,
        "contig": contig,
        "length": length,
        "redesign_motif_sidechains": redesign_motif_sidechains,
        "is_non_loopy": is_non_loopy,
        "ligand": ligand_name,
        "select_fixed_atoms": fixed_atoms,
    }
}
if infer_ori_strategy is not None:
    payload[design_name]["infer_ori_strategy"] = infer_ori_strategy

with open(rfd3_json, "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote RFD3 input ->", rfd3_json)

### Write the RFD3 submit script

Teaching defaults are tiny (`diffusion_batch_size=2`, `n_batches=1` → 2 backbones) so the
shared queue stays usable. Scale up only if the queue is empty.

In [ ]:
queue        = "c27666"
job_name     = "rfd3"
cores        = 4
gpu_spec     = "num=1:mode=exclusive_process"
time_limit   = "1:00"
mem          = "10GB"
CKPT_PATH    = "/dtu/projects/dbl/foundry/ckpt/rfd3_latest.ckpt"

diffusion_batch_size = 2    # backbones per batch
n_batches            = 1    # total backbones = diffusion_batch_size * n_batches

script_path = os.path.join(submit_dir, "rfd3.sh")

script = f"""#!/bin/sh
#BSUB -q {queue}
#BSUB -J {job_name}
#BSUB -n {cores}
#BSUB -gpu "{gpu_spec}"
#BSUB -W {time_limit}
#BSUB -R "rusage[mem={mem}]"
#BSUB -R "span[hosts=1]"
#BSUB -o {logs_dir}/%J.out
#BSUB -e {logs_dir}/%J.err

mkdir -p {logs_dir} {diffusion_out_dir}
module load cuda/12.4
source /dtu/projects/dbl/foundry/miniforge3/etc/profile.d/conda.sh
conda activate /dtu/projects/dbl/foundry/miniforge3/envs/rfd3

export RFD3_PATH="/dtu/projects/dbl/foundry"
export PYTHONPATH="${{RFD3_PATH}}:${{PYTHONPATH:-}}"
export DISABLE_CUEQUIVARIANCE=true   # skip cuEquivariance import (crashes on c27666 A100 driver via pynvml NVML_NOT_SUPPORTED); optimization only
# export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   # reduce fragmentation on the ~20GB MIG slice - TODO: needed?

rfd3 design \\
    out_dir="{diffusion_out_dir}" \\
    inputs="{rfd3_json}" \\
    ckpt_path="{CKPT_PATH}" \\
    diffusion_batch_size={diffusion_batch_size} \\
    n_batches={n_batches} \\
    low_memory_mode=True \\
    inference_sampler.step_scale=1.5 \\
    inference_sampler.gamma_0=0.2

echo "Completed at $(date)"
"""

with open(script_path, "w") as f:
    f.write(script)

print("Wrote", script_path)
print("\nSubmit in a terminal:\n  bsub < " + script_path)
print("Check progress:  bstat        (wait until the rfd3 job is gone)")

### Process RFD3 outputs

Each backbone is written as a `.cif.gz` plus a `.json` of geometry metrics (and the
`diffused_index_map` LigandMPNN needs to locate the catalytic residues). Run the cells
below **after** the RFD3 job finishes to gather and eyeball those metrics.

In [ ]:
json_rfd3_dir    = diffusion_out_dir
rfd3_metrics_csv = Path(scores_dir) / "rfd3_metrics_with_json_path.csv"
# -----------------------------

json_paths = sorted(glob.glob(os.path.join(json_rfd3_dir, "*.json")))
if not json_paths:
    raise SystemExit(f"No JSON files found in {json_rfd3_dir!r} — run RFD3 jobs first.")

rows = []
all_keys = set(["json_path"])

for jp in json_paths:
    with open(jp, "r") as f:
        data = json.load(f)

    metrics = data.get("metrics", {})
    row = {"json_path": os.path.abspath(jp)}
    for k, v in metrics.items():
        if k in ("diffused_com", "fixed_com") and isinstance(v, (list, tuple)) and len(v) == 3:
            row[f"{k}_x"], row[f"{k}_y"], row[f"{k}_z"] = v
            all_keys.update({f"{k}_x", f"{k}_y", f"{k}_z"})
        else:
            row[k] = v
            all_keys.add(k)
    rows.append(row)

fieldnames = ["json_path"] + sorted(k for k in all_keys if k != "json_path")

with open(rfd3_metrics_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print(f"Wrote {len(rows)} rows -> {rfd3_metrics_csv}")
print("Columns:", fieldnames)

In [ ]:
df = pd.read_csv(rfd3_metrics_csv)
df.head(10)

In [ ]:
metrics = [
    "alanine_content",
    "glycine_content",
    "helix_fraction",
    "loop_fraction",
    "sheet_fraction",
    "max_ca_deviation",
    "n_chainbreaks",
    "n_clashing.interresidue_clashes_w_backbone",
    "n_clashing.interresidue_clashes_w_sidechain",
    "non_loop_fraction",
    "radius_of_gyration",
]

# Keep only columns that exist (safety)
metrics = [m for m in metrics if m in df.columns]

print("Plotting:", metrics)

n_metrics = len(metrics)
n_cols = 4
n_rows = math.ceil(n_metrics / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

def plot_hist(ax, series, title):
    s = series.dropna()
    if s.empty:
        ax.set_title(f"{title} (no data)")
        ax.axis("off")
        return
    ax.hist(s, bins=30)
    ax.set_title(title)
    ax.set_ylabel("count")

for ax, col in zip(axes, metrics):
    plot_hist(ax, df[col], col)

# Turn off unused axes
for ax in axes[len(metrics):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Optional: filter out broken backbones

Backbones with chain breaks or steric clashes fold unreliably and waste downstream
(shared) GPU time. We drop them here using the geometry metrics RFD3 already wrote — no
heavy structure-parsing dependencies needed. Passing designs are copied into a
`passing/` subfolder; LigandMPNN picks them up automatically.

Skip this section to keep **all** backbones — LigandMPNN falls back to the full set if no
backbone passes (e.g. at the tiny default of 2 backbones).

In [ ]:
bb_clash = "n_clashing.interresidue_clashes_w_backbone"
sc_clash = "n_clashing.interresidue_clashes_w_sidechain"

mask = (df["n_chainbreaks"] == 0)
if bb_clash in df.columns:
    mask &= (df[bb_clash] == 0)
if sc_clash in df.columns:
    mask &= (df[sc_clash] == 0)

df_filt = df[mask].copy()

print(f"Total designs: {len(df)}")
print(f"Passing designs (no chain breaks / clashes): {len(df_filt)}")
df_filt.head()

In [ ]:
PATH_COL = "json_path"
filter_name = "passing"

filtered_dir = Path(diffusion_out_dir) / filter_name
filtered_dir.mkdir(parents=True, exist_ok=True)

def copy_design(json_path: str, dst_dir: Path) -> int:
    """Copy one design's JSON and its matching CIF/CIF.GZ (same stem) into dst_dir."""
    jp = Path(json_path)
    if not jp.exists():
        return 0
    copied = 0
    dst_json = dst_dir / jp.name
    if not dst_json.exists():
        shutil.copy2(jp, dst_json)
        copied += 1
    stem = jp.stem
    for cif in (jp.with_name(stem + ".cif"), jp.with_name(stem + ".cif.gz")):
        if cif.exists():
            dst_cif = dst_dir / cif.name
            if not dst_cif.exists():
                shutil.copy2(cif, dst_cif)
                copied += 1
    return copied

total_copied = missing = 0
unique_paths = df_filt[PATH_COL].dropna().unique()
for pth in unique_paths:
    if not isinstance(pth, str) or not os.path.exists(pth):
        print(f"[MISSING] {pth}")
        missing += 1
        continue
    total_copied += copy_design(pth, filtered_dir)

print(f"Passing designs: {len(unique_paths)}")
print(f"Copied files: {total_copied}")
print(f"Output folder: {filtered_dir}")

## 2. LigandMPNN — design sequences

LigandMPNN writes amino-acid sequences onto each backbone **conditioned on the substrate**,
so the designed pocket complements the ligand. The catalytic residues stay fixed: for every
backbone we read its `diffused_index_map` (written by RFD3) to find where the catalytic
residues landed in the designed numbering, and pass those to `--fixed_residues`.

This builds one command per backbone and submits them as a single LSF **job array**. We do
**not** filter the LigandMPNN output — every sequence goes forward to RF3.

In [ ]:
seed              = 42
batch_size        = 2     # sequences per backbone
number_of_batches = 1     # total sequences per backbone = batch_size * number_of_batches
chains_to_design  = "A"   # enzyme chain
ckpt = "/dtu/projects/dbl/LigandMPNN/25.3.1-0//model_params/ligandmpnn_v_32_010_25.pt"
MPNN_RUN = "/dtu/projects/dbl/LigandMPNN/25.3.1-0/run.py"

cmds = f"{cmds_dir}/mpnn.cmds"
filtered = sorted(glob.glob(f"{diffusion_out_dir}/passing/*.cif.gz"))
structures = filtered if filtered else sorted(glob.glob(f"{diffusion_out_dir}/*.cif.gz"))
print(f"Using {len(structures)} backbone(s) "
      + ("(clash/break-filtered)" if filtered else "(all backbones — no filter applied)"))
if not structures:
    raise SystemExit(f"No backbones (*.cif.gz) in {diffusion_out_dir} — run RFD3 first.")

n_written = 0
with open(cmds, "w") as f:
    for structure in structures:
        bn = os.path.basename(structure).replace(".cif.gz", "")
        this_out = f"{mpnn_out_dir}/{bn}"
        os.makedirs(this_out, exist_ok=True)

        # Map catalytic residues to their position in the designed backbone
        side_json = structure.replace(".cif.gz", ".json")
        if not os.path.exists(side_json):
            print(f"[skip] no sidecar JSON for {bn}")
            continue
        data = json.load(open(side_json))
        index_map = data.get("diffused_index_map", {})
        fixed_resis = " ".join(index_map[r] for r in catres if r in index_map)

        cmd = (f"python {MPNN_RUN} --seed {seed} --pdb_path \"{structure}\" "
               f"--out_folder \"{this_out}\" --batch_size {batch_size} "
               f"--number_of_batches {number_of_batches} --model_type ligand_mpnn "
               f"--checkpoint_ligand_mpnn \"{ckpt}\" --chains_to_design \"{chains_to_design}\" "
               f"--fixed_residues \"{fixed_resis}\"")
        f.write(cmd + "\n")
        n_written += 1

if n_written == 0:
    raise SystemExit("No MPNN commands written (missing sidecar JSONs?).")

n_tasks = sum(1 for _ in open(cmds))
sub_script = jupyter_utils.make_sub_script(
    cmds, n_task=n_tasks, group_size=10, mem="10G", queue="c27666",
    job_name="mpnn", cores=4, time_limit="2:00",
    python_path="/dtu/projects/dbl/LigandMPNN/25.3.1-0/",
    env="/dtu/projects/dbl/LigandMPNN/25.3.1-0//miniforge3/envs/ligandmpnn_env",
)
print(f"\n{n_tasks} MPNN command(s).")
print("Submit:\n  bsub < " + sub_script)

## 3. RF3 — fold and score each design

RF3 folds each designed sequence together with the substrate to predict the full enzyme +
ligand complex. One RF3 job per backbone (it folds all that backbone's sequences in one go).

The substrate is supplied to RF3 as chemistry: a **SMILES** string for our non-CCD
PET-mimic ligand (or a CCD code if your ligand is standard).

### Define the substrate and build RF3 input JSONs

In [ ]:
# Substrate for RF3. RF3 needs chemistry to place the ligand:
#   - non-CCD ligand -> give a SMILES string (our case)
#   - standard ligand -> set ccd_ligand=True and put the CCD code in ligand_code
ccd_ligand  = False
ligand_code = "pt1"                              # CCD code, used only if ccd_ligand=True
smiles_str  = "COC(=O)c1ccc(C(=O)OCCO)cc1"       # PET-monomer mimic; used if ccd_ligand=False

if ccd_ligand:
    ligand_entry = {"ccd_code": ligand_code}
elif smiles_str:
    ligand_entry = {"smiles": smiles_str}
else:
    raise SystemExit("Ligand not in CCD — provide a SMILES string (smiles_str).")

ligand_entry

In [ ]:
rf3_json_dir = Path(configs_dir) / "rf3"
rf3_json_dir.mkdir(parents=True, exist_ok=True)

def parse_fasta(path):
    header, chunks = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks)
                header, chunks = line[1:], []
            else:
                chunks.append(line)
        if header is not None:
            yield header, "".join(chunks)

fa_files = sorted(glob.glob(os.path.join(mpnn_out_dir, "**", "seqs", "*.cif.gz.fa"),
                            recursive=True))
if not fa_files:
    raise SystemExit(f"No MPNN FASTAs under {mpnn_out_dir} — run MPNN first.")

written = 0
for fa in fa_files:
    name = os.path.basename(fa).split(".")[0]   # backbone name
    designs = []
    for i, (header, seq) in enumerate(parse_fasta(fa)):
        if i == 0:
            continue   # first record is the original RFD3 backbone (alanine fill) — skip
        chain_a = seq.split(":", 1)[0]
        designs.append({
            "name": f"{name}_{i}",
            "components": [
                {"seq": chain_a, "chain_id": "A"},
                ligand_entry,
            ],
        })
    if not designs:
        continue
    with open(rf3_json_dir / f"{name}.json", "w") as out:
        json.dump(designs, out, indent=2)
    written += 1

print(f"Wrote {written} RF3 input JSON(s) -> {rf3_json_dir}")

### Write the RF3 submit script (job array)

In [ ]:
queue       = "c27666"
job_name    = "rf3"
time_limit  = "2:00"
mem         = "10GB"
gpu_spec    = "num=1:mode=exclusive_process"
cores       = 4
group_size  = 10                    # backbone JSONs folded per array task
rf3_release = "/dtu/projects/dbl/rf3/release"
ckpt_path   = "/dtu/projects/dbl/rf3/release/ckpt/rf3_latest.pt"
num_steps   = 50                    # 50 for speed; raise toward 200 for production quality

json_files = sorted(Path(rf3_json_dir).glob("*.json"))
if not json_files:
    raise SystemExit(f"No RF3 JSONs in {rf3_json_dir}")

cmds_path = Path(cmds_dir) / "rf3.cmds"
lines = []
for jf in json_files:
    out_dir = Path(rf3_out_dir) / jf.stem
    lines.append(
        f"rf3 fold inference_engine=rf3 inputs={jf} out_dir={out_dir} "
        f"ckpt_path={ckpt_path} num_steps={num_steps} "
        f"annotate_b_factor_with_plddt=True early_stopping_plddt_threshold=0"
    )
cmds_path.write_text("\n".join(lines) + "\n")

n_arrays = math.ceil(len(lines) / group_size)
script = f"""#!/bin/sh
#BSUB -q {queue}
#BSUB -J {job_name}[1-{n_arrays}]
#BSUB -n {cores}
#BSUB -gpu "{gpu_spec}"
#BSUB -W {time_limit}
#BSUB -R "rusage[mem={mem}]"
#BSUB -R "span[hosts=1]"
#BSUB -o {logs_dir}/%J_%I.out
#BSUB -e {logs_dir}/%J_%I.err

mkdir -p {logs_dir}
module load cuda/12.4
export RF3_PATH="{rf3_release}"
source {rf3_release}/activate_env.sh
export PYTHONPATH="${{RF3_PATH}}:${{PYTHONPATH:-}}"
export DISABLE_CUEQUIVARIANCE=true   # same workaround as RFD3 (harmless if RF3 does not read it)

CMDS_FILE={cmds_path}
GROUP_SIZE={group_size}
START=$(( (LSB_JOBINDEX - 1) * GROUP_SIZE ))
END=$(( START + GROUP_SIZE ))
i=0
while IFS= read -r cmd; do
    if [ "$i" -ge "$START" ] && [ "$i" -lt "$END" ]; then
        echo "Running: $cmd"
        eval "$cmd"
    fi
    i=$((i+1))
done < "$CMDS_FILE"
echo "Done at $(date)"
"""

sub = Path(submit_dir) / "rf3.sh"
sub.write_text(script)
print(f"{len(lines)} RF3 fold job(s) in {n_arrays} array task(s).")
print("Submit:\n  bsub < " + str(sub))

### Score and plot RF3 results

Run after the RF3 jobs finish. We parse the per-design `.score` files. For an enzyme the
relevant chains are the **enzyme (`A_1`)** and the **substrate (`B_1`)**, so the
enzyme–substrate "interface" metrics tell you how confidently RF3 docks the ligand in the
pocket. Key metrics per design:

- **`enzyme_plddt`** — confidence of the enzyme fold (0–1; higher is better).
- **`ligand_min_pae`** — enzyme↔substrate interface PAE (Å; **lower** is better).
- **`ligand_ipsae`** — enzyme↔substrate interface ipSAE (0–1; **higher** = better-defined pocket).

> We reuse `gather_rf3_metrics` from `lib/`: it is written in binder terms, so here the
> "binder" is the enzyme chain and the "target" is the substrate. We rename the columns
> below for clarity.

In [ ]:
# Chain IDs in RF3 output:  A_1 = enzyme  |  B_1 = substrate (ligand)
ENZYME = "A_1"
LIGAND = "B_1"

OUT_CSV = f"{scores_dir}/rf3_gathered_metrics.csv"

df_rf3 = gather_rf3_metrics(
    parent=rf3_out_dir,
    binder=ENZYME,
    target_f=LIGAND,
    target_g=LIGAND,
    out_csv=OUT_CSV,
)

# Rename binder-centric columns to enzyme terms for readability
df_rf3 = df_rf3.rename(columns={
    "best_binder_plddt": "enzyme_plddt",
    "AF_best_min_pae":   "ligand_min_pae",
    "AF_ipsae_at_best":  "ligand_ipsae",
})
print(f"{len(df_rf3)} designs scored")
df_rf3.head()

In [ ]:
# Thresholds -- shared across the filtering and selection cells below
PLDDT_CUT = 0.80   # enzyme_plddt   > PLDDT_CUT (0-1)
PAE_CUT   = 8.0    # ligand_min_pae < PAE_CUT   (A)
IPSAE_CUT = 0.50   # ligand_ipsae   > IPSAE_CUT (0-1)

df = df_rf3
x = df["ligand_min_pae"].to_numpy(float)
y = df["enzyme_plddt"].to_numpy(float)
c = df["ligand_ipsae"].to_numpy(float)
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 13,
    "axes.linewidth": 1.1, "figure.dpi": 150,
})
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x[mask], y[mask], c=c[mask], s=20, alpha=0.85, linewidths=0,
                cmap="viridis", vmin=0, vmax=1)
ax.axvline(PAE_CUT,   color="k", lw=0.9, ls="--", alpha=0.6)
ax.axhline(PLDDT_CUT, color="k", lw=0.9, ls="--", alpha=0.6)
ax.set_xlabel("Enzyme-substrate PAE (A)  [lower = better]")
ax.set_ylabel("Enzyme pLDDT  [higher = better]")
ax.set_title(f"RF3 validation - PAE vs pLDDT  (n={len(df)})")
ax.set_xlim(left=0)
ax.set_ylim(0, 1.0)
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
fig.colorbar(sc, ax=ax).set_label("substrate ipSAE  [higher = better]")
fig.tight_layout()
plt.show()

In [ ]:
# Ranked enzyme pLDDT across all designs
ranked = df_rf3.sort_values("enzyme_plddt", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(ranked.index, ranked["enzyme_plddt"], alpha=0.8, s=20)
ax.axhline(PLDDT_CUT, color="tomato", linestyle="--", label=f"threshold ({PLDDT_CUT})")
ax.set_xlabel("design rank")
ax.set_ylabel("enzyme_plddt")
ax.set_title("RF3 pLDDT - ranked designs")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Collect your best designs

Designs passing all three thresholds are saved to `best_designs/` (the best RF3 model CIF
plus a scored CSV). These are your candidates to inspect in PyMOL/ChimeraX — check that the
catalytic triad and substrate sit as intended in the pocket.

In [ ]:
import gzip as _gz

pass_mask = (
    (df_rf3["enzyme_plddt"]   > PLDDT_CUT) &
    (df_rf3["ligand_min_pae"] < PAE_CUT) &
    (df_rf3["ligand_ipsae"]   > IPSAE_CUT)
)
df_best = df_rf3[pass_mask].copy()
print(f"{len(df_best)} / {len(df_rf3)} designs pass thresholds "
      f"(pLDDT>{PLDDT_CUT}, PAE<{PAE_CUT}, ipSAE>{IPSAE_CUT})")

best_dir = Path(best_designs_dir)
best_dir.mkdir(parents=True, exist_ok=True)

copied_cif, missing = 0, []
for _, row in df_best.iterrows():
    did  = row["design_id"]
    bidx = int(row["best_batch_idx"]) if "best_batch_idx" in row and pd.notna(row["best_batch_idx"]) else 0
    sf   = Path(row["score_file"])

    cif_src = sf.parent / f"{did}_model_{bidx}.cif.gz"      # best RF3 model
    dst_cif = best_dir / f"{did}_model_{bidx}.cif"          # decompressed
    if cif_src.exists():
        with _gz.open(cif_src, "rb") as gz_in, open(dst_cif, "wb") as out:
            out.write(gz_in.read())
        copied_cif += 1
    else:
        missing.append(str(cif_src))

print(f"Copied {copied_cif} CIF -> {best_dir}")
if missing:
    print(f"Missing {len(missing)} source file(s); first few:")
    for m in missing[:10]:
        print("  ", m)

best_csv = best_dir / "best_designs_metrics.csv"
df_best.to_csv(best_csv, index=False)
print(f"Metrics -> {best_csv}")
df_best[["design_id", "enzyme_plddt", "ligand_min_pae", "ligand_ipsae"]]